# Fine-tuning Llama 3.2 with ORPO (Unsloth) on free Colab

This notebook follows the recipe from [Dolmaa24/llm-course](https://github.com/Dolmaa24/llm-course) (a fork of Maxime Labonne's LLM course), specifically the course's **Supervised Fine-Tuning / Preference Alignment** sections:

- Fine-tuning framework: [Unsloth](https://docs.unsloth.ai/) (QLoRA, 2-5x faster, fits a free T4)
- Alignment method: **ORPO** (Odds Ratio Preference Optimization) — combines SFT + preference alignment in a single stage, following the course's ["Fine-tune Llama 3 with ORPO"](https://mlabonne.github.io/blog/posts/2024-04-19_Fine_tune_Llama_3_with_ORPO.html) notebook, adapted here to the smaller **Llama 3.2 3B** model so it comfortably trains on a free Colab GPU.
- Dataset: [`mlabonne/orpo-dpo-mix-40k`](https://huggingface.co/datasets/mlabonne/orpo-dpo-mix-40k) — a public preference dataset (prompt / chosen / rejected), same one used in the course's ORPO and DPO notebooks.

**Runtime setup:** In Colab, go to `Runtime > Change runtime type > T4 GPU` (free tier) before running.

**How to use this notebook:**
1. Run cells top to bottom.
2. Everything is driven by the `CONFIG` cell below — swap the model, dataset size, or LoRA settings there.
3. To go from ORPO back to plain SFT or to DPO instead, see the note at the bottom.


## 1. Install dependencies

Unsloth's official Colab install (pulls in a compatible `trl`, `peft`, `accelerate`, `bitsandbytes`).

In [ ]:
%%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl>=0.9.6" peft accelerate bitsandbytes


In [ ]:
# Sanity check: confirm a GPU is attached (Runtime > Change runtime type > T4 GPU)
import torch
assert torch.cuda.is_available(), "No GPU detected. Go to Runtime > Change runtime type > T4 GPU."
print(torch.cuda.get_device_name(0))


## 2. Config

Everything below reads from this cell. Swap `MODEL_NAME` for the 1B variant if you want an even faster run, or point `DATASET_NAME` at your own preference dataset later.

In [ ]:
# --- Model ---
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"   # swap to "unsloth/Llama-3.2-1B-Instruct-bnb-4bit" for a faster/smaller run
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

# --- LoRA ---
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# --- Dataset ---
DATASET_NAME = "mlabonne/orpo-dpo-mix-40k"
NUM_SAMPLES = 2000     # subset for a quick demo run; set to None to use the full 40k (slower)
MAX_PROMPT_LENGTH = 1024

# --- ORPO training ---
OUTPUT_DIR = "outputs"
NUM_TRAIN_EPOCHS = 1
PER_DEVICE_BATCH_SIZE = 2
GRAD_ACCUMULATION_STEPS = 4
LEARNING_RATE = 8e-6
ORPO_BETA = 0.1        # lambda weight on the odds-ratio term

# --- Saving ---
SAVE_LORA_DIR = "llama3.2-orpo-lora"
MERGED_DIR = "llama3.2-orpo-merged-16bit"
PUSH_TO_HUB = False                     # set True to push, and fill in HF_REPO_ID below
HF_REPO_ID = "your-username/llama3.2-orpo-demo"


## 3. Load the base model (4-bit) with Unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,  # auto-detect (bfloat16 on Ampere+/T4 fallback to float16)
)


## 4. Attach LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth's long-context-friendly checkpointing
    random_state=3407,
)
model.print_trainable_parameters()


## 5. Load and format the preference dataset

`mlabonne/orpo-dpo-mix-40k` stores `chosen` and `rejected` as chat-format message lists (system/user/assistant turns). ORPOTrainer needs plain-text `prompt`, `chosen`, and `rejected` columns, so we render each with the model's own chat template and split off the final assistant turn as the completion.

In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset(DATASET_NAME, split="train")
if NUM_SAMPLES is not None:
    raw_dataset = raw_dataset.shuffle(seed=3407).select(range(min(NUM_SAMPLES, len(raw_dataset))))

print(raw_dataset)
print(raw_dataset[0])


In [ ]:
def to_prompt_chosen_rejected(example):
    # `chosen`/`rejected` are lists of {"role": ..., "content": ...} turns that share
    # every turn except the final assistant reply.
    chosen_turns = example["chosen"]
    rejected_turns = example["rejected"]

    prompt_turns = chosen_turns[:-1]  # everything up to (not including) the final assistant answer
    chosen_answer = chosen_turns[-1]
    rejected_answer = rejected_turns[-1]

    prompt_text = tokenizer.apply_chat_template(
        prompt_turns, tokenize=False, add_generation_prompt=True
    )
    chosen_text = chosen_answer["content"] + tokenizer.eos_token
    rejected_text = rejected_answer["content"] + tokenizer.eos_token

    return {"prompt": prompt_text, "chosen": chosen_text, "rejected": rejected_text}

formatted_dataset = raw_dataset.map(
    to_prompt_chosen_rejected,
    remove_columns=raw_dataset.column_names,
)
print(formatted_dataset[0]["prompt"][:500])
print("---CHOSEN---")
print(formatted_dataset[0]["chosen"][:300])
print("---REJECTED---")
print(formatted_dataset[0]["rejected"][:300])


## 6. Train with ORPO

ORPO jointly optimizes a supervised loss on `chosen` and an odds-ratio term that pushes `chosen` above `rejected` — no separate SFT stage or reference model needed (unlike DPO).

In [ ]:
from trl import ORPOConfig, ORPOTrainer

orpo_config = ORPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    beta=ORPO_BETA,
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=MAX_PROMPT_LENGTH,
    logging_steps=10,
    save_strategy="epoch",
    optim="adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="none",
)

trainer = ORPOTrainer(
    model=model,
    args=orpo_config,
    train_dataset=formatted_dataset,
    processing_class=tokenizer,
)


In [ ]:
trainer_stats = trainer.train()
print(trainer_stats)


## 7. Save the fine-tuned model

In [ ]:
# LoRA adapters only (small, fast to save)
model.save_pretrained(SAVE_LORA_DIR)
tokenizer.save_pretrained(SAVE_LORA_DIR)

# Optional: merge LoRA into the base weights and save a full 16-bit model
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")

if PUSH_TO_HUB:
    from huggingface_hub import login
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))  # store your token in Colab's Secrets panel as HF_TOKEN
    model.push_to_hub_merged(HF_REPO_ID, tokenizer, save_method="merged_16bit")


## 8. Quick inference check

In [ ]:
FastLanguageModel.for_inference(model)  # enables Unsloth's fast inference path

messages = [
    {"role": "user", "content": "What are three tips for staying focused while studying?"},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
)
print(tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True))


## Next steps

- **Use your own data instead of the demo dataset**: build a `datasets.Dataset` with `prompt` / `chosen` / `rejected` string columns (or `chosen`/`rejected` message-list columns like the demo dataset, reusing the formatting cell above) and set `DATASET_NAME`/loading accordingly.
- **Plain instruction fine-tuning (SFT) instead of preference alignment**: replace section 5's `chosen`/`rejected` formatting with a single `text` column per the course's [Fine-tune Llama 3.1 with Unsloth](https://mlabonne.github.io/blog/posts/2024-07-29_Finetune_Llama31.html) notebook, and swap `ORPOTrainer`/`ORPOConfig` for `trl`'s `SFTTrainer`/`SFTConfig`.
- **DPO instead of ORPO**: keep the same `prompt`/`chosen`/`rejected` dataset shape, but swap in `trl`'s `DPOTrainer`/`DPOConfig` (note DPO needs a frozen reference model, which costs more VRAM than ORPO).
- **Scale up**: raise `NUM_SAMPLES` toward the full 40k, increase `NUM_TRAIN_EPOCHS`, or move to Colab Pro / an A100 for a larger base model (e.g. Llama 3.1 8B, matching the course's flagship example).
- **Evaluate**: run the fine-tuned model through the course's [LLM AutoEval](https://github.com/mlabonne/llm-autoeval) notebook to benchmark it.
